In [ ]:
# Install dependencies
!pip install transformers sentencepiece tqdm -q


In [ ]:
# Clone data repo
!git clone https://github.com/Aditya-dom/legaldocsvalution.git /content/legaldocsvalution -q


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os, torch
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("nsi319/legal-led-base-16384")
model = AutoModelForSeq2SeqLM.from_pretrained("nsi319/legal-led-base-16384")
model = model.to(device)

save_parent = "/content/legal_led_outputs"
read_dir = "/content/legaldocsvalution/dataset/indian_data/ind_text"
os.makedirs(save_parent, exist_ok=True)


In [ ]:
avg_len = 5459  # pre-computed average token length across the 93 documents

for file_ in tqdm(os.listdir(read_dir)):
    filepath = os.path.join(read_dir, file_)
    with open(filepath, "r", errors="ignore") as f:
        text = f.read()

    input_tokenized = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        max_length=avg_len,
        truncation=True
    ).input_ids.to(device)

    for summary_fraction in [0.3]:
        save_dir = os.path.join(save_parent, f"summary_{summary_fraction}")
        os.makedirs(save_dir, exist_ok=True)

        summary_ids = model.generate(
            input_tokenized,
            num_beams=2,
            no_repeat_ngram_size=3,
            length_penalty=2,
            min_length=int((summary_fraction - 0.2) * avg_len),
            max_length=int(summary_fraction * avg_len)
        )
        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        with open(os.path.join(save_dir, file_), "w") as out:
            out.write(summary)

print("Done! Outputs in /content/legal_led_outputs/")


In [ ]:
# Download results
import shutil
from google.colab import files
shutil.make_archive("/content/legal_led_summaries", "zip", "/content/legal_led_outputs")
files.download("/content/legal_led_summaries.zip")
